# Deep Learning 025 — Dropout, Part 2: Practice

Companion notebook to the lesson. Lesson 024 established *why* dropout works. This one is
about using it: where the layer goes, what `p` should be, and the one asymmetry that causes
most dropout bugs — **it is on during training and off during inference.**

| Question | Measured below |
|---|---|
| what does `p` actually trade? | the gap narrows to `p ≈ 0.5`, then training accuracy starts paying |
| why does test time need no mask? | inverted dropout keeps the expected activation fixed |
| what if you forget the `1/(1-p)` scaling? | every activation arrives **2× too large**, and accuracy may not notice |
| does it help a model that is *under*fitting? | **no** — it removes capacity that was not spare |

`numpy` only.

In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = make_moons(n_samples=500, noise=0.28, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.6, random_state=0, stratify=y)
s = StandardScaler().fit(X_tr)
X_tr, X_te = s.transform(X_tr), s.transform(X_te)
y_tr, y_te = y_tr.reshape(-1, 1) * 1.0, y_te.reshape(-1, 1) * 1.0

def relu(z):    return np.maximum(0, z)
def d_relu(z):  return (z > 0) * 1.0
def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -30, 30)))

def train(p_drop=0.0, hidden=64, epochs=3000, lr=0.3, seed=1, inverted=True):
    r = np.random.default_rng(seed)
    W1 = r.normal(size=(2, hidden)) * np.sqrt(2 / 2);      b1 = np.zeros(hidden)
    W2 = r.normal(size=(hidden, 1)) * np.sqrt(2 / hidden); b2 = np.zeros(1)
    for _ in range(epochs):
        z1 = X_tr @ W1 + b1
        h = relu(z1)
        if p_drop:
            keep = (r.random(h.shape) > p_drop) * 1.0
            mask = keep / (1 - p_drop) if inverted else keep
        else:
            mask = 1.0
        hd = h * mask
        out = sigmoid(hd @ W2 + b2)
        d_out = (out - y_tr) / len(X_tr)
        gW2, gb2 = hd.T @ d_out, d_out.sum(0)
        d_h = (d_out @ W2.T) * mask * d_relu(z1)
        gW1, gb1 = X_tr.T @ d_h, d_h.sum(0)
        W1 -= lr * gW1; b1 -= lr * gb1; W2 -= lr * gW2; b2 -= lr * gb2
    return W1, b1, W2, b2

def predict(prm, X, scale=1.0):
    W1, b1, W2, b2 = prm
    return sigmoid((relu(X @ W1 + b1) * scale) @ W2 + b2)

def acc(prm, X, t, scale=1.0):
    return float(((predict(prm, X, scale) > 0.5) == (t > 0.5)).mean())

## Part A — Sweeping `p`

`p` is the probability of **dropping** a unit. Both extremes fail, and they fail in opposite
directions — which is the same shape as every other regularisation dial in this course.

In [ ]:
base_tr = acc(train(p_drop=0.0, hidden=64), X_tr, y_tr)
print(f"{'p':>6}{'train':>9}{'test':>9}{'gap':>8}   reading")
for p in (0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9, 0.95, 0.98):
    prm = train(p_drop=p, hidden=64)
    tr, te = acc(prm, X_tr, y_tr), acc(prm, X_te, y_te)
    note = ("no regularisation" if p == 0 else
            "capacity is being paid for" if base_tr - tr > 0.03 else
            "workable")
    print(f"{p:>6.2f}{tr:>9.3f}{te:>9.3f}{tr - te:>8.3f}   {note}")

Two things to notice, and the second is a caveat about this dataset rather than about
dropout.

The **gap** column is the one dropout is aimed at, and it narrows — 0.038 with no dropout,
0.020 at `p = 0.3`, with the best test accuracy in the same region. Past `p ≈ 0.9` the
training accuracy starts falling faster than the gap closes, which is capacity being paid
for rather than bought.

And the failure at the top end is *gentle* here — even `p = 0.98` still scores 0.857. That
is a property of the problem, not a general truth: two input dimensions and a smooth
boundary do not need many units, so throwing away 98% of a 64-unit layer still leaves
enough. On a task with real structure to represent, high `p` collapses the model properly.
**Always sweep `p` on your own data rather than trusting a number from someone else's.**

The published rules of thumb are still the right place to start:

| Where | Typical `p` |
|---|---|
| input layer | 0.1 – 0.2 — you are discarding raw data, so be gentle |
| hidden layers of a plain ANN | 0.1 – 0.5 |
| convolutional layers | 0.4 – 0.5 |
| above 0.5 | not recommended |
| output layer | never |

## Part B — The train/test asymmetry, and why inverted dropout exists

At training time a unit is present with probability `1-p`, so the expected input to the next
layer is `(1-p)` times what it would be with everything on. At test time nothing is dropped.
**Something has to compensate, or the two phases disagree.**

There are two ways to fix it, and modern frameworks all pick the same one.

In [ ]:
r = np.random.default_rng(0)
h = np.abs(r.normal(size=(20000, 64))) + 0.1        # stand-in for post-ReLU activations
p = 0.5

plain = h * ((r.random(h.shape) > p) * 1.0)                 # classic dropout
inv = h * (((r.random(h.shape) > p) * 1.0) / (1 - p))       # INVERTED dropout

print(f"{'':<34}{'mean activation':>18}")
print(f"{'no dropout (i.e. test time)':<34}{h.mean():>18.4f}")
print(f"{'classic dropout, train time':<34}{plain.mean():>18.4f}")
print(f"{'inverted dropout, train time':<34}{inv.mean():>18.4f}")
print(f"\nclassic  is off by a factor of {h.mean() / plain.mean():.2f}  = 1/(1-p)")
print(f"inverted matches to {abs(inv.mean() / h.mean() - 1):.4f} relative error")

- **Classic dropout** leaves training activations `(1-p)` too small, so *inference* must
  multiply every weight by `(1-p)` to compensate. That means test time is different code.
- **Inverted dropout** divides by `(1-p)` during training instead. Expected activation is
  then unchanged, and **inference needs no adjustment at all** — you simply stop applying
  the mask.

Every framework uses the inverted form, which is why `model.eval()` in PyTorch or
`training=False` in Keras is all that is required.

## Part C — What forgetting the scaling actually costs

The most common dropout bug is training with the classic form and predicting as if it were
inverted. Here is the size of that mistake.

In [ ]:
prm_inv = train(p_drop=0.5, inverted=True)
prm_plain = train(p_drop=0.5, inverted=False)

print(f"{'trained with':<24}{'predicted with':<26}{'test acc':>9}{'mean p':>9}")
rows = [("inverted dropout", "no mask (correct)", prm_inv, 1.0),
        ("classic dropout", "no mask (WRONG)", prm_plain, 1.0),
        ("classic dropout", "weights x (1-p) (right)", prm_plain, 0.5)]
for tl, pl, prm, sc in rows:
    pr = predict(prm, X_te, sc)
    print(f"{tl:<24}{pl:<26}{acc(prm, X_te, y_te, sc):>9.3f}{pr.mean():>9.3f}")

good = predict(prm_plain, X_te, 0.5)
bad = predict(prm_plain, X_te, 1.0)
print(f"\nthe bug, measured on the PROBABILITIES rather than the labels:")
print(f"   mean |p_wrong - p_right| = {np.abs(bad - good).mean():.4f}")
print(f"   predictions that flip    = {((bad > 0.5) != (good > 0.5)).mean():.1%}")
print(f"   mean confidence          = {good.mean():.3f} correct vs {bad.mean():.3f} wrong")

The middle row is the bug, and **the accuracy column barely notices it** — which is exactly
what makes it dangerous. It does not crash, it does not warn, and on an easy binary task a
uniform doubling of the pre-activation rarely pushes a prediction across the 0.5 threshold.

Look at the probabilities instead. Every activation reaching the output layer is twice the
size the network was trained for, so the model's *confidence* is wrong even where its
*label* is right — and any downstream use of those numbers (a calibrated threshold, an
expected-cost calculation, a ranking) inherits the error silently. On a harder task, or a
deeper network where the factor compounds per layer, the labels move too.

## Part D — Dropout is not a general-purpose improvement

Lesson 021's diagnosis comes first, always. Dropout is an **overfitting** fix, and applying
it to a model that is underfitting makes things worse rather than better.

In [ ]:
print(f"{'network':<20}{'p':>6}{'train':>9}{'test':>9}{'change in test':>16}")
for hidden, label in ((2, "tiny (2 units)"), (64, "large (64 units)")):
    base = train(p_drop=0.0, hidden=hidden)
    b_te = acc(base, X_te, y_te)
    for p in (0.0, 0.5):
        prm = train(p_drop=p, hidden=hidden)
        te = acc(prm, X_te, y_te)
        delta = "" if p == 0 else f"{te - b_te:+.3f}"
        print(f"{label:<20}{p:>6.1f}{acc(prm, X_tr, y_tr):>9.3f}{te:>9.3f}{delta:>16}")

Read the last column. On the two-unit network dropout removes capacity the model did not
have to spare, and test accuracy goes **down**. On the 64-unit network there is capacity to
spare, and it goes **up**.

**Same technique, opposite sign, and the only thing that changed was whether the model had
too much capacity or too little.** Which is why lesson 021's diagnosis is not a formality.

## In Keras

```python
from tensorflow.keras.layers import Dropout

model = keras.Sequential([
    keras.layers.Dense(128, activation="relu"),
    Dropout(0.5),                       # p is the DROP probability
    keras.layers.Dense(64, activation="relu"),
    Dropout(0.3),                       # per layer, not global
    keras.layers.Dense(1, activation="sigmoid"),   # never after the output
])
```

Keras applies inverted dropout and switches it off automatically for `predict` and
`evaluate`, so there is nothing to remember at inference — which is precisely the payoff of
Part B.

## Try it yourself

1. Put dropout on the *input* layer at `p = 0.2` as well. Does it help here, and would you
   expect it to on 2-D data?
2. Increase `epochs` to 20,000 with `p = 0.5`. Does dropout eventually overfit too, or does
   it hold?
3. Re-run Part A with `hidden=256`. Does the workable band for `p` move?
4. Implement the classic (non-inverted) form correctly — scaling the *weights* at inference
   — and confirm it matches the inverted form to three decimals. Then argue for why the
   inverted version is still the better engineering choice.